**Geochemistry Biplot App for Bruker Results.csv Files**

Jupyter Notebook Version
N. Tripcevich 2026, CC BY-SA 4.0  
[More Information Online](https://github.com/arf-berkeley/bruker-xrf-ppm-plot)

For basic use click here, then proceed through this notebook cell-by-cell by pressing Shift-Return on your keyboard. Follow the instructions provided to upload your .csv file and view the data.

The Python in this live notebook can be edited and run again.

In [1]:
%%capture
%pip install plotly ipywidgets
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.colors import DEFAULT_PLOTLY_COLORS
from IPython.display import display
import ipywidgets as widgets
import io

**Select the Results.csv table from your Bruker analysis**

Browse to a copy of the __Results.csv__ file typically found in Bruker/Data/Results.csv

The code below parses **all method segments** from the Results.csv file.
A filter step in the next cell lets you choose which method and batch to visualise.

In [7]:
# Cell 2 - CSV Import - supports both local and hosted environments
from io import StringIO

study_import          = None
study_import_filtered = None

# ── environment detection ─────────────────────────────────────────────────
def is_local():
    try:
        import tkinter as tk
        root = tk.Tk()
        root.destroy()
        return True
    except Exception:
        return False

# ── parser ────────────────────────────────────────────────────────────────
def parse_results_csv(content_str):
    """
    Parse a Bruker Results.csv containing multiple calibration segments.
    Each segment begins with a header row whose first cell is 'File #'.
    Returns a DataFrame with all segments stacked, plus a '_batch' column.
    """
    # Normalise line endings
    content_str = content_str.replace('\r\n', '\n').replace('\r', '\n')
    lines = [l for l in content_str.split('\n') if l.strip()]

    print(f'  Non-empty lines : {len(lines)}')
    print(f'  Line 1 preview  : {repr(lines[0][:100])}')

    # Find every row where the first CSV field is literally 'File #'
    segment_starts = [
        i for i, line in enumerate(lines)
        if line.split(',')[0].strip().strip('"') == 'File #'
    ]
    print(f'  Segments found  : {len(segment_starts)} '
          f'(at lines {segment_starts[:8]}{"..." if len(segment_starts)>8 else ""})')

    if not segment_starts:
        raise ValueError('No "File #" header row found — is this a Bruker Results.csv?')

    # Parse each segment independently with pd.read_csv
    frames = []
    for idx, start in enumerate(segment_starts, start=1):
        end           = segment_starts[idx] if idx < len(segment_starts) else len(lines)
        segment_lines = lines[start:end]
        if len(segment_lines) < 2:
            continue                          # header-only block, skip
        try:
            df_seg = pd.read_csv(
                StringIO('\n'.join(segment_lines)),
                dtype=str,
                skipinitialspace=True
            )
        except Exception as e:
            print(f'  Segment {idx} skipped: {e}')
            continue

        df_seg.dropna(how='all', inplace=True)
        df_seg.dropna(axis=1, how='all', inplace=True)
        if df_seg.empty:
            continue

        df_seg['_batch'] = idx
        frames.append(df_seg)
        print(f'  Segment {idx:>3}: {len(df_seg):>4} rows | '
              f'method={df_seg["Method"].dropna().unique().tolist() if "Method" in df_seg.columns else "?"}')

    if not frames:
        raise ValueError('No data found after parsing all segments.')

    df = pd.concat(frames, ignore_index=True, join='outer')
    df.replace({'< LOD': None, 'None': None, '': None}, inplace=True)

    print(f'\n✓ Total rows : {len(df)}')
    print(f'  Batches    : {sorted(df["_batch"].unique().tolist())}')
    if 'Method' in df.columns:
        print(f'  Methods    : {df["Method"].dropna().unique().tolist()}')
    return df

# ── filter UI ─────────────────────────────────────────────────────────────
# ── filter UI ─────────────────────────────────────────────────────────────
def build_filter_ui():
    global study_import_filtered
    study_import_filtered = study_import.copy()

    # ── helpers
    def clean_series(col):
        return (study_import[col]
                .dropna()
                .astype(str)
                .str.strip()
                .replace('', pd.NA)
                .dropna())

    def get_unique(col):
        if col not in study_import.columns:
            return []
        return sorted(clean_series(col).unique().tolist())

    def batches_for(application):
        df = study_import.copy()
        if application != 'All applications':
            df = df[df['Application'].astype(str).str.strip() == application]
        return ['All'] + [str(b) for b in sorted(df['_batch'].dropna().unique())]

    # ── widgets
    all_applications = ['All applications'] + get_unique('Application')
    default_app      = all_applications[1] if len(all_applications) > 1 else 'All applications'

    app_dd = widgets.Dropdown(
        options=all_applications,
        value=default_app,
        description='Application:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='360px')
    )
    batch_dd = widgets.Dropdown(
        options=batches_for(default_app),
        value='All',
        description='Batch:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='180px')
    )
    apply_btn = widgets.Button(
        description='Apply Filter',
        button_style='primary',
        icon='filter',
        layout=widgets.Layout(width='150px')
    )
    summary_out = widgets.Output()
    filter_out  = widgets.Output()

    # ── callbacks
    def on_app_change(change):
        new_batches      = batches_for(change['new'])
        batch_dd.options = new_batches
        batch_dd.value   = 'All'
        with summary_out:
            summary_out.clear_output(wait=True)
            df = study_import.copy()
            if change['new'] != 'All applications':
                df = df[df['Application'].astype(str).str.strip() == change['new']]
            batches = sorted(df['_batch'].unique().tolist())
            dates   = pd.to_datetime(df['DateTime'], errors='coerce').dropna()
            d_min   = dates.min().strftime('%m-%d-%Y') if not dates.empty else '?'
            d_max   = dates.max().strftime('%m-%d-%Y') if not dates.empty else '?'
            print(f'  "{change["new"]}" → {len(df)} rows')
            print(f'  Batch(es) : {batches}')
            print(f'  Date range: {d_min} – {d_max}')

    def on_apply(btn):
        global study_import_filtered
        filter_out.clear_output(wait=True)
        with filter_out:
            df         = study_import.copy()
            sel_app    = app_dd.value
            sel_batch  = batch_dd.value

            if sel_app != 'All applications':
                df = df[df['Application'].astype(str).str.strip() == sel_app]
            if sel_batch != 'All':
                df = df[df['_batch'] == int(sel_batch)]

            df = df.reset_index(drop=True)
            study_import_filtered = df

            if df.empty:
                print(f'No rows for application="{sel_app}" batch="{sel_batch}"')
                # Show what IS available to help diagnose
                print(f'\nApplications in file : {get_unique("Application")}')
                print(f'Batches in file      : {sorted(study_import["_batch"].unique().tolist())}')
                dates_all = pd.to_datetime(
                    study_import['DateTime'], errors='coerce'
                ).dropna()
                print(f'Full date range      : {dates_all.min()} – {dates_all.max()}')
                return

            # Parse dates properly for display
            dates = pd.to_datetime(df['DateTime'], errors='coerce').dropna()
            d_min = dates.min().strftime('%m-%d-%Y %H:%M') if not dates.empty else '?'
            d_max = dates.max().strftime('%m-%d-%Y %H:%M') if not dates.empty else '?'
            f_min = df['File #'].dropna().min() if 'File #' in df.columns else '?'
            f_max = df['File #'].dropna().max() if 'File #' in df.columns else '?'

            print(f'✓ {len(df)} rows kept')
            print(f'  Application : {sel_app}')
            print(f'  Batch       : {sel_batch}')
            print(f'  File # range: {f_min} – {f_max}')
            print(f'  Date range  : {d_min} – {d_max}')
            print('\nNow run Cell 3 → Cell 4 → Cell 5')

    app_dd.observe(on_app_change, names='value')
    apply_btn.on_click(on_apply)

    # ── layout
    display(widgets.HTML('<b>Filter by Application and Batch</b>'))
    display(widgets.HTML(
        '<span style="color:grey;font-size:12px">'
        'Choose an Application — Batch will update automatically. '
        'Then click Apply Filter.</span>'
    ))
    display(widgets.VBox([
        widgets.HBox([app_dd, batch_dd, apply_btn]),
        summary_out,
        filter_out
    ]))

    # prime the UI
    on_app_change({'new': app_dd.value})
    on_apply(None)

# ── file loading ──────────────────────────────────────────────────────────
if is_local():
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    study_path = filedialog.askopenfilename(
        title='Select Results.csv',
        filetypes=[('CSV files', '*.csv'), ('All files', '*.*')]
    )
    root.destroy()
    if study_path:
        with open(study_path, 'r', encoding='utf-8', errors='replace') as f:
            study_import = parse_results_csv(f.read())
        build_filter_ui()
    else:
        print('No file selected.')

else:
    upload_widget    = widgets.FileUpload(accept='.csv', multiple=False)
    load_btn         = widgets.Button(
        description='Load Data',
        button_style='success',
        icon='check',
        disabled=True
    )
    status_lbl       = widgets.Label('Upload a Bruker XRF Results.csv file')
    filter_area      = widgets.Output()

    def _on_upload_change(change):
        load_btn.disabled = not bool(upload_widget.value)
        if upload_widget.value:
            status_lbl.value = 'File ready — click Load Data'

    def _on_load(btn):
        global study_import
        try:
            raw          = upload_widget.value[0]['content'].tobytes()
            content      = raw.decode('utf-8', errors='replace')
            study_import = parse_results_csv(content)
            status_lbl.value      = f'✓ Loaded {study_import.shape[0]} rows'
            load_btn.disabled     = True
            load_btn.description  = 'Loaded'
            with filter_area:
                filter_area.clear_output(wait=True)
                build_filter_ui()
        except Exception as e:
            status_lbl.value = f'Error: {e}'

    upload_widget.observe(_on_upload_change, names='value')
    load_btn.on_click(_on_load)

    display(widgets.VBox([
        widgets.Label('Upload Results.csv:'),
        upload_widget,
        load_btn,
        status_lbl,
        filter_area
    ]))

  Non-empty lines : 308
  Line 1 preview  : 'File #,DateTime,Operator,Name,ID,Field1,Field2,Application,Method,ElapsedTime,Alloy 1,Match Qual 1,A'
  Segments found  : 15 (at lines [0, 2, 4, 6, 42, 45, 57, 81]...)
  Segment   1:    1 rows | method=['PrecMetals']
  Segment   2:    1 rows | method=['PrecMetals']
  Segment   3:    1 rows | method=?
  Segment   4:   35 rows | method=['Obsidian 3mm']
  Segment   5:    2 rows | method=['PrecMetalsSmallSample', 'PrecMetals']
  Segment   6:   11 rows | method=['PrecMetals', 'PrecMetalsSmallSample']
  Segment   7:   23 rows | method=['Obsidian 3mm']
  Segment   8:   22 rows | method=['PrecMetals', 'PrecMetalsSmallSample']
  Segment   9:   16 rows | method=['Obsidian 3mm']
  Segment  10:    4 rows | method=['PrecMetalsSmallSample', 'PrecMetals']
  Segment  11:   91 rows | method=['Obsidian 3mm']
  Segment  12:    3 rows | method=['PrecMetals', 'PrecMetalsSmallSample']
  Segment  13:    4 rows | method=['PrecMetals', 'PrecMetalsSmallSample']
  Seg

HTML(value='<b>Filter by Application and Batch</b>')

HTML(value='<span style="color:grey;font-size:12px">Choose an Application — Batch will update automatically. T…

**Filter by Method / Batch**

The instrument writes multiple calibration segments into one file. Use the controls below to choose which **Method** and **Batch** to pass into the cleaning and plotting steps.

- *Method* – e.g. `Obsidian 3mm`, `PrecMetals`, `PrecMetalsSmallSample`  
- *Batch* – sequential segment number within the file (1 = first segment)  
- Leave **Batch** on `All` to keep every batch that matches the selected method.

***[Click Here to Continue]***

__Clean up Bruker data__

Cleaning data includes removing the following: elemental error columns, Alloy, Match Qual columns, Multiplier, Cal Check, Operator, Field 1&2. This script also replaces Below Detection Limits LOD with 0.

In [8]:
# Cell 3 - Data cleaning
# Now reads from study_import_filtered (set by Cell 2b) instead of
# study_import directly, so only the chosen method/batch is processed.

_source = (
    study_import_filtered
    if 'study_import_filtered' in dir() and study_import_filtered is not None
    else study_import
)

if _source is None:
    print('Please upload Results.csv and apply a filter before continuing.')
else:
    # Metadata columns to keep
    META_COLS = ['File #', 'DateTime', 'Name', 'Application', 'Method', 'ElapsedTime']

    # Non-element columns to drop
    NON_ELEMENT_COLS = [
        'Alloy 1', 'Match Qual 1', 'Alloy 2', 'Match Qual 2',
        'Alloy 3', 'Match Qual 3', 'Multiplier', 'Cal Check',
        'Operator', 'Field1', 'Field2', 'ID', '_batch'
    ]

    # Drop error columns and non-element columns
    drop_cols = [
        c for c in _source.columns
        if 'Err' in c or c in NON_ELEMENT_COLS
    ]
    keep_cols = [c for c in _source.columns if c not in drop_cols]

    study = _source[keep_cols].copy()

    # Replace below detection limit values with zero
    study = study.replace('< LOD', 0)

    # Ensure correct types
    string_cols  = [c for c in ['Name', 'Application', 'Method'] if c in study.columns]
    numeric_cols = [c for c in study.columns if c not in string_cols + ['DateTime']]
    study[string_cols]  = study[string_cols].astype('string')
    study[numeric_cols] = study[numeric_cols].apply(pd.to_numeric, errors='coerce')
    study['DateTime']   = pd.to_datetime(study['DateTime'], errors='coerce')

    # Identify element columns (numeric, excluding metadata)
    element_cols = [c for c in numeric_cols if c not in ['File #', 'ElapsedTime']]

    # Convert element values from percent to PPM and round to 1 decimal place
    study[element_cols] = (study[element_cols] * 10000).round(1)

    # Drop rows and columns with all NaN
    study.dropna(axis=1, how='all', inplace=True)
    study.dropna(how='all', inplace=True)

    print('Dataset Headers')
    print('Study: ' + str(study.columns.tolist()))
    print(f'Rows after dropping NaN: {len(study)}')

Dataset Headers
Study: ['File #', 'DateTime', 'Application', 'Method', 'ElapsedTime', 'Co', 'Zn', 'Ag', 'Sb', 'Ti', 'Cu', 'Sn', 'Au', 'Pb']
Rows after dropping NaN: 11


**Display Data Table**

Run the next cell to view the data table before viewing a biplot.

In [9]:
# Cell 4 - Data Table
if study is None:
    print('Please complete data cleaning before continuing')
else:
    display(widgets.HTML('<b>Results Table</b>'))
    display_df = study.copy()
    display_df['DateTime'] = display_df['DateTime'].dt.strftime('%m/%d/%Y %H:%M')
    display_df = display_df.rename(columns={'ElapsedTime': 'Elapsed'})

    non_element = ['File #', 'DateTime', 'Name', 'Application', 'Method', 'Elapsed']
    text_cols   = [c for c in ['DateTime', 'Name', 'Application', 'Method']
                   if c in display_df.columns]
    element_cols = [
        c for c in display_df.columns
        if c not in non_element
        and pd.api.types.is_numeric_dtype(display_df[c])
    ]

    display(display_df.style
        .format({col: '{:.1f}' for col in element_cols})
        .set_properties(**{'text-align': 'right', 'font-size': '12px'})
        .set_properties(subset=text_cols, **{'text-align': 'left'})
        .set_table_styles([{
            'selector': 'th',
            'props': [('text-align', 'center'), ('font-weight', 'bold')]
        }])
        .hide(axis='index')
    )

HTML(value='<b>Results Table</b>')

File #,DateTime,Application,Method,Elapsed,Co,Zn,Ag,Sb,Ti,Cu,Sn,Au,Pb
1061,05/21/2026 15:47,Precious Metals 2,PrecMetals,15.000000,0.0,nan,924356.0,nan,nan,75644.0,nan,nan,nan
1062,05/21/2026 15:48,Precious Metals 2,PrecMetals,15.000000,nan,nan,nan,nan,nan,873820.0,126180.0,nan,nan
1063,05/21/2026 15:50,Precious Metals 2,PrecMetalsSmallSample,15.000000,0.0,1265.0,874987.0,nan,1958.0,118320.0,nan,nan,3470.0
1064,05/21/2026 15:51,Precious Metals 2,PrecMetalsSmallSample,15.000000,0.0,nan,957705.0,2326.0,nan,34323.0,nan,nan,5646.0
1065,05/21/2026 15:52,Precious Metals 2,PrecMetals,15.000000,0.0,160536.0,787804.0,2379.0,nan,45626.0,nan,nan,3655.0
1066,05/21/2026 15:54,Precious Metals 2,PrecMetals,15.000000,0.0,nan,662404.0,nan,nan,333636.0,nan,nan,3960.0
1067,05/21/2026 15:57,Precious Metals 2,PrecMetalsSmallSample,15.000000,0.0,nan,728427.0,nan,nan,269251.0,nan,nan,2323.0
1068,05/21/2026 15:58,Precious Metals 2,PrecMetals,15.000000,0.0,nan,569361.0,nan,nan,425328.0,nan,2550.0,2761.0
1069,05/21/2026 15:59,Precious Metals 2,PrecMetalsSmallSample,15.000000,0.0,nan,969130.0,3174.0,595.0,14761.0,nan,nan,12341.0
1070,05/21/2026 15:59,Precious Metals 2,PrecMetalsSmallSample,15.000000,0.0,nan,972778.0,3497.0,nan,9088.0,nan,nan,14636.0


In [11]:
# Cell 5 - Biplot
if study is None:
    print('Please complete data cleaning before continuing')
else:
    # Exclude non-element columns from axis dropdowns
    NON_ELEMENT_COLS = ['File #', 'DateTime', 'Name', 'Application',
                        'Method', 'ElapsedTime', 'Elapsed', '_batch', '_method']
    elements_present = [
        c for c in study.columns
        if c not in NON_ELEMENT_COLS
        and pd.api.types.is_numeric_dtype(study[c])
    ]

    if len(elements_present) < 2:
        print(f'Not enough numeric element columns to plot. Found: {elements_present}')
    else:
        x_dropdown = widgets.Dropdown(
            options=elements_present,
            value='Sr' if 'Sr' in elements_present else elements_present[0],
            description='X Axis:',
            style={'description_width': 'initial'}
        )
        y_dropdown = widgets.Dropdown(
            options=elements_present,
            value='Rb' if 'Rb' in elements_present else elements_present[1],
            description='Y Axis:',
            style={'description_width': 'initial'}
        )

        plot_output = widgets.Output()

        def update_plot(change):
            with plot_output:
                plot_output.clear_output(wait=True)
                x = x_dropdown.value
                y = y_dropdown.value

                # Build hover_data only from columns that exist
                # and are not already the x or y axis
                desired_hover = ['File #', 'Name', 'DateTime']
                hover_data = [
                    c for c in desired_hover
                    if c in study.columns and c != x and c != y
                ]

                # Ensure Name exists and has usable values for coloring
                if 'Name' in study.columns:
                    # Fill blank/NaN names with a placeholder so every
                    # point gets a visible legend entry
                    plot_df = study.copy()
                    plot_df['Name'] = (
                        plot_df['Name']
                        .astype(str)
                        .str.strip()
                        .replace({'': '(no name)', 'nan': '(no name)', '<NA>': '(no name)'})
                    )
                    color_col = 'Name'
                else:
                    plot_df  = study.copy()
                    color_col = None

                try:
                    # Get sorted unique names so legend order is consistent
                    if color_col:
                        name_order = sorted(plot_df['Name'].dropna().unique().tolist())
                    else:
                        name_order = []

                    fig = px.scatter(
                        plot_df,
                        x=x,
                        y=y,
                        color=color_col,
                        category_orders={'Name': name_order},
                        hover_data=hover_data,
                        title=f'{y} vs {x} Biplot',
                        labels={
                            x: f'{x} (PPM)',
                            y: f'{y} (PPM)',
                            'Name': 'Sample Name'
                        }
                    )

                    fig.update_traces(marker=dict(size=8, opacity=0.85))

                    fig.update_layout(
                        height=600,
                        hovermode='closest',
                        legend=dict(
                            title=dict(text='Sample Name', font=dict(size=13)),
                            itemsizing='constant',
                            bordercolor='lightgrey',
                            borderwidth=1,
                            bgcolor='rgba(255,255,255,0.85)',
                            x=1.02,
                            xanchor='left',
                            y=1,
                            yanchor='top'
                        ),
                        margin=dict(r=180)   # make room for legend outside plot
                    )

                    fig.show()

                except Exception as e:
                    print(f'Plot error: {e}')
                    print(f'  x={x}, y={y}')
                    print(f'  study columns: {study.columns.tolist()}')
                    print(f'  study dtypes:\n{study.dtypes}')

        x_dropdown.observe(update_plot, names='value')
        y_dropdown.observe(update_plot, names='value')

        display(widgets.VBox([
            widgets.HBox([x_dropdown, y_dropdown]),
            plot_output
        ]))

        update_plot(None)

In [12]:
# Cell 6 - Export dataset as CSV
from IPython.display import display, HTML
if study is None:
    print('Please complete data cleaning before continuing')
else:
    export_output = widgets.Output()
    export_btn    = widgets.Button(
        description='Export CSV', button_style='success', icon='download'
    )

    def on_export(btn):
        with export_output:
            export_output.clear_output(wait=True)
            if is_local():
                import tkinter as tk
                from tkinter import filedialog
                root = tk.Tk()
                root.withdraw()
                root.attributes('-topmost', True)
                save_path = filedialog.asksaveasfilename(
                    title='Save CSV',
                    defaultextension='.csv',
                    filetypes=[('CSV files', '*.csv'), ('All files', '*.*')],
                    initialfile=(
                        f'Bruker_Results_export_'
                        f'{study["DateTime"].max().strftime("%Y%m%d")}.csv'
                    )
                )
                root.destroy()
                if save_path:
                    study.to_csv(save_path, index=False)
                    print(f'✓ Exported {len(study)} rows to {save_path}')
                else:
                    print('Export cancelled')
            else:
                import base64
                csv_str  = study.to_csv(index=False)
                b64      = base64.b64encode(csv_str.encode()).decode()
                filename = (
                    f'Bruker_Results_export_'
                    f'{study["DateTime"].max().strftime("%Y%m%d")}.csv'
                )
                html = (
                    f'<a download="{filename}" '
                    f'href="data:text/csv;base64,{b64}">'
                    f'Click here to download {filename}</a>'
                )
                display(HTML(html))

    export_btn.on_click(on_export)
    display(widgets.HTML('<b>Export the cleaned CSV of these values?</b>'))
    display(widgets.VBox([export_btn, export_output]))

HTML(value='<b>Export the cleaned CSV of these values?</b>')